# El Quirófano de datos

## 1. Contexto y Preparación: 
La empresa quiere predecir el éxito de futuras aplicaciones móviles. Han extraído un volcado de datos de la Google Play Store, pero el pipeline de Machine Learning rechaza el archivo. Su misión como ingenieros de datos es sanear los valores faltantes aplicando reglas de negocio estrictas.

## 2. Instrucciones Técnicas: 
Abran su Jupyter Notebook. Pueden descargar el dataset googleplaystore.csv de su repositorio o usar este enlace crudo (si lo hospedas en tu GitHub):

In [1]:
import pandas as pd
import numpy as np
 
 
# Reemplaza con la URL de tu repositorio
url = "../datasets/googleplaystore.csv"
df = pd.read_csv(url)

df.sample(10)

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
2189,Avokiddo Emotions,FAMILY,4.6,73,12M,"1,000+",Paid,$2.99,Everyone,Education;Pretend Play,"May 21, 2018",1.7.0,4.0.3 and up
1918,Helix Jump,GAME,4.2,1500999,33M,"100,000,000+",Free,0,Everyone,Action,"April 9, 2018",1.0.6,4.1 and up
10367,FG SPINNER,GAME,NaN,2,36M,10+,Free,0,Everyone,Board,"April 6, 2018",0.90,4.0.3 and up
1192,Simple Recipes,FOOD_AND_DRINK,4.7,3803,15M,"500,000+",Free,0,Everyone,Food & Drink,"February 22, 2018",1.6,4.0 and up
9773,ER Doctor City Emergency - Surgery City Doc FREE,FAMILY,3.9,1909,84M,"100,000+",Free,0,Teen,Educational,"June 13, 2018",2.1,4.0.3 and up
1116,Credit Sesame,FINANCE,4.3,25166,11M,"1,000,000+",Free,0,Everyone,Finance,"July 25, 2018",3.1.16,4.1 and up
1382,My Diet Coach - Weight Loss Motivation & Tracker,HEALTH_AND_FITNESS,4.4,141163,Varies with device,"10,000,000+",Free,0,Everyone,Health & Fitness,"June 11, 2018",Varies with device,Varies with device
1536,How to sew & make the easiest pattern,LIBRARIES_AND_DEMO,NaN,110,10M,"10,000+",Free,0,Everyone,Libraries & Demo,"November 5, 2017",1.0,4.0 and up
3589,We learn children's verses in kindergarten,PARENTING,NaN,3248,6.5M,"100,000+",Free,0,Everyone,Parenting;Education,"December 20, 2017",2.10,4.0 and up
4776,iLauncher OS 12 Pro - Phone X,PERSONALIZATION,NaN,3,9.6M,50+,Paid,$2.99,Everyone,Personalization,"June 24, 2018",1.0.1,4.1 and up


## 3. Preguntas de Análisis (Para responder con código y celdas Markdown):

### Pregunta A: 
Ejecuten df.isnull().sum(). Identificarán que la columna Rating tiene casi 1,500 valores nulos, mientras que Type y Android Ver tienen solo 1 o 2. ¿Por qué es arquitectónicamente seguro eliminar las filas donde falta Android Ver, pero sería un error fatal eliminar las filas donde falta Rating?

In [2]:
df.isnull().sum()

App                  0
Category             0
Rating            1474
Reviews              0
Size                 0
Installs             0
Type                 1
Price                0
Content Rating       1
Genres               0
Last Updated         0
Current Ver          8
Android Ver          3
dtype: int64

In [3]:
# df.groupby("Android Ver")["Android Ver"].count()

In [4]:
# df.groupby("Type")["Type"].count()

Rating es un dato de tipo MNAR (missing not at random), su ausencia nos da información, mientras que la version de android o el Type, son son o más tipo MCAR (missing completively at random), el type puede ser Free y Paid, y la versiones de android son bastantes, pero es más como un error de sistema

### Pregunta B: 
Escriban la instrucción exacta usando dropna() pasándole el parámetro subset=['Android Ver', 'Type'] para eliminar únicamente las 3 filas que tienen esos nulos. Verifiquen que el total de registros en el DataFrame apenas disminuyó.

In [5]:
print("antes: ",len(df))
dropna= df.dropna(subset=["Android Ver","Type"])
print("despues: ",len(dropna))



antes:  10841
despues:  10837


antes:  10841
despues:  10837

### Pregunta C: 
Rellenar los 1,500 nulos de Rating con el promedio global es un error, ya que las apps de la categoría FAMILY (Familia) no se califican igual que las de DATING (Citas). Utilizando la lógica de la semana pasada, escriban un bloque combinando .groupby('Category') y .transform() para rellenar los NaN de la columna Rating usando la mediana de su propia categoría.

In [6]:
# stratified imputation
df_c = df.copy()
df_c["Rating"] = df_c["Rating"].fillna(df_c.groupby("Category")["Rating"].transform("median"))
print("BEFORE: \n-------------\n",df.isnull().sum(),"\n------------------")
print("AFTER: \n-------------\n",df_c.isnull().sum())


BEFORE: 
-------------
 App                  0
Category             0
Rating            1474
Reviews              0
Size                 0
Installs             0
Type                 1
Price                0
Content Rating       1
Genres               0
Last Updated         0
Current Ver          8
Android Ver          3
dtype: int64 
------------------
AFTER: 
-------------
 App               0
Category          0
Rating            0
Reviews           0
Size              0
Installs          0
Type              1
Price             0
Content Rating    1
Genres            0
Last Updated      0
Current Ver       8
Android Ver       3
dtype: int64


### Pregunta D: 
Revisen la columna Size (Tamaño de la app). Notarán que no tiene nulos matemáticos (NaN), pero muchas filas dicen "Varies with device". Para un modelo matemático, ese texto es basura indigerible. Escriban una instrucción usando df['Size'].replace(...) para convertir exactamente esa frase de texto en un nulo matemático real usando np.nan de NumPy. ¿Por qué es mejor convertirlo a NaN en esta etapa en lugar de ponerle un tamaño falso como "0 MB"?


In [7]:
df.groupby("Size")["Size"].count()

Size
1,000+                   1
1.0M                     7
1.1M                    32
1.2M                    41
1.3M                    35
                      ... 
98M                     16
992k                     1
994k                     1
99M                     39
Varies with device    1695
Name: Size, Length: 462, dtype: int64

In [8]:
df_d = df.copy()
df_d["Size"]= df_d["Size"].replace("Varies with device", np.nan)
df_d.groupby("Size")["Size"].count()

Size
1,000+     1
1.0M       7
1.1M      32
1.2M      41
1.3M      35
          ..
986k       1
98M       16
992k       1
994k       1
99M       39
Name: Size, Length: 461, dtype: int64

Ponerle un falso tamaño sería un sesgo, es mejor usar NaN para poder hacerle skip más facil los datos.